# Existe relação entre velocidade/direção do vento e chuvas intensas?

Comparação descritiva das seis horas anteriores às horas de chuva intensa (> 10 mm/h) e fraca (0 < chuva ≤ 2 mm/h), com dados de Recife de 2019 a 2020. O limite de chuva intensa segue o projeto.

Cada curva mantém a mesma amostra nos seis pontos, usando janelas completas por variável e pareamento por data e hora real. Valores ausentes, não finitos, chuva/velocidade negativas e direções fora de 0 a 360 graus são descartados. A direção usa média circular e exclui calmaria (velocidade zero). Os grupos são definidos pela chuva em T-0; as horas anteriores também podem ter chuva.

`n` conta horas-alvo, não episódios independentes. As médias não controlam horário, sazonalidade ou outras condições meteorológicas e não quantificam incerteza. A comparação não estabelece causalidade, significância estatística nem poder preditivo. Direções médias podem ocultar dispersão.

Execute a célula para recriar o PNG e o SVG editável em `figuras/`.


In [ ]:
import csv
from datetime import datetime, timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if not (ROOT / "dados/df_recife.csv").exists():
    ROOT = ROOT.parent

def numero(texto):
    try:
        valor = float(texto.replace(",", "."))
        return valor if np.isfinite(valor) else np.nan
    except ValueError:
        return np.nan

def media_circular(graus):
    vetor = np.mean(np.exp(1j * np.deg2rad(graus)), axis=0)
    return np.where(np.abs(vetor) > 1e-8, np.rad2deg(np.angle(vetor)) % 360, np.nan)

# A média entre 350° e 10° deve apontar para o norte, nunca para 180°.
assert min(float(media_circular([350, 10])), 360 - float(media_circular([350, 10]))) < 1e-8
assert np.isnan(numero("")) and numero("1,5") == 1.5

dados = {}
with (ROOT / "dados/df_recife.csv").open(encoding="utf-8", newline="") as arquivo:
    for linha in csv.DictReader(arquivo):
        try:
            hora = datetime.strptime(linha["Data"] + " " + linha["Hora UTC"], "%Y-%m-%d %H%M UTC")
        except ValueError:
            continue
        chuva = numero(linha["PRECIPITAÇÃO TOTAL, HORÁRIO (mm)"])
        velocidade = numero(linha["VENTO, VELOCIDADE HORARIA (m/s)"])
        direcao = numero(linha["VENTO, DIREÇÃO HORARIA (gr) (° (gr))"])
        assert hora not in dados, "Verificar horários duplicados antes de analisar."
        dados[hora] = (chuva if chuva >= 0 else np.nan,
                       velocidade if velocidade >= 0 else np.nan,
                       direcao % 360 if 0 <= direcao <= 360 and velocidade > 0 else np.nan)

# Pareamento por horário real. Cada curva usa somente janelas completas.
# As horas-alvo podem pertencer ao mesmo episódio de chuva.
horizontes = range(6, 0, -1)
amostras = [[[], []], [[], []]]
for hora, (chuva, _, _) in dados.items():
    grupo = 0 if chuva > 10 else 1 if 0 < chuva <= 2 else None
    if grupo is None:
        continue
    janela = [dados.get(hora - timedelta(hours=h), (np.nan,) * 3) for h in horizontes]
    for metrica in range(2):
        valores = [registro[metrica + 1] for registro in janela]
        if np.isfinite(valores).all():
            amostras[metrica][grupo].append(valores)

cores = ["#287db2", "#19b3c2"]
nomes = ["Antes de chuva intensa (> 10 mm/h)", "Antes de chuva fraca (0 < chuva ≤ 2 mm/h)"]
plt.rcParams.update({"font.size": 14, "axes.spines.top": False, "axes.spines.right": False})
fig, eixos = plt.subplots(2, 1, figsize=(10, 10))
fig.suptitle("Vento antes da chuva em Recife", fontsize=20, weight="bold", y=0.975)
fig.text(0.5, 0.935, "Recife • 2019 a 2020 • Seis horas antes da chuva", ha="center", fontsize=14)
resumo = []
for metrica, ax in enumerate(eixos):
    for grupo in range(2):
        matriz = np.asarray(amostras[metrica][grupo])
        assert matriz.ndim == 2 and matriz.shape[1] == 6
        medias = np.mean(matriz, axis=0) if metrica == 0 else media_circular(matriz)
        assert np.isfinite(medias).all()
        ax.plot(range(6), medias, color=cores[grupo], marker=["o", "s"][grupo], linewidth=2.5,
                markersize=7, label=f"{nomes[grupo]} • n={len(matriz)}")
        for x, y in enumerate(medias):
            rotulo = f"{y:.1f}".replace(".", ",") + ("°" if metrica else "")
            ax.annotate(rotulo, (x, y), xytext=(0, (12 if grupo == 1 else -23) if metrica == 0 else (12 if grupo == 0 else -23)),
                        textcoords="offset points", ha="center", color=cores[grupo], fontsize=14, weight="bold")
        resumo.append({"metrica": ["velocidade", "direcao"][metrica], "grupo": nomes[grupo],
                       "n": len(matriz), "medias": medias.tolist()})
    ax.set_title(["Velocidade média do vento", "Direção média do vento"][metrica], loc="left", fontsize=17, weight="bold", pad=14)
    ax.set_xticks(range(6), [f"T-{h}h" for h in horizontes])
    ax.set_ylabel(["Velocidade (m/s)", "Direção (graus)"][metrica])
    ax.grid(alpha=0.25)
    ax.set_axisbelow(True)
    ax.margins(x=0.06, y=0.4)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.14), fontsize=12, frameon=False)
fig.text(0.5, 0.085, "Direção: 180° = sul • 225° = sudoeste • 270° = oeste", ha="center", fontsize=12)
fig.text(0.5, 0.027, "Fonte: dados/df_recife.csv • n = horas-alvo com seis horas anteriores válidas\n"
         "Direção: média circular, sem calmaria • Horas podem pertencer ao mesmo episódio\n"
         "Comparação descritiva: não demonstra causalidade nem capacidade de previsão.", ha="center", fontsize=10)
fig.subplots_adjust(top=0.87, bottom=0.21, left=0.12, right=0.96, hspace=0.72)
(ROOT / "figuras").mkdir(exist_ok=True)
fig.savefig(ROOT / "figuras/vento_chuva_intensa.png", dpi=160)
fig.savefig(ROOT / "figuras/vento_chuva_intensa.svg")
for linha in resumo:
    print(linha)
plt.show()
